# Train Orion's "hey orion" wake word

This notebook trains a custom openWakeWord model for **"hey orion"** on a free Colab GPU,
using openWakeWord's synthetic-data pipeline. It's Phase 6 of
[Orion](https://github.com/shahwazreza/orion): see `docs/wake-word.md` there for what to do with the result.

**What happens**

1. A multi-speaker TTS model (Piper, LibriTTS voices) says "hey orion" tens of thousands of times,
   with varied speakers, speeds and intonation. It also says similar-sounding phrases to learn
   what *not* to react to.
2. Each clip is mixed with real room echoes (MIT impulse responses) and background audio (AudioSet).
3. A small network is trained against ~2,000 hours of pre-computed general audio (ACAV100M
   features), aiming for at most 0.2 false activations per hour.
4. The result is exported as `hey_orion.onnx` (about 200 KB) for you to download.

**Privacy:** everything here is synthetic or public. Nothing from your computer, your voice, or
Orion's data is uploaded.

**Time and space:** on a free T4 GPU, about 1.5 to 3 hours with the default settings, and about
40 GB of disk (mostly the 16 GB negative-audio features). Run the cells in order:
*Runtime > Run all* works. If Colab disconnects, re-run from the top: finished steps are skipped.

**Before you start:** *Runtime > Change runtime type > T4 GPU*.

In [ ]:
!nvidia-smi -L || echo "No GPU found: Runtime > Change runtime type > T4 GPU (CPU works but takes many hours)"

## 1. Settings
The defaults are a good balance. Lower `N_SAMPLES` / `STEPS` only for a quick trial run.

In [ ]:
TARGET_PHRASES = ["hey orion"]
MODEL_NAME = "hey_orion"

N_SAMPLES = 50_000        # synthetic "hey orion" clips for training (20k minimum, more is better)
N_SAMPLES_VAL = 5_000     # held out for validation
STEPS = 50_000            # training steps
TARGET_FP_PER_HOUR = 0.2  # aim for at most this many false activations per hour

# Phrases that sound alike but must NOT wake Orion. Add names of people you live with.
NEGATIVE_PHRASES = ["hey ryan", "hey brian", "o'brien", "hey arya", "hey lauren", "hey jarvis",
                    "hey siri", "orange", "oreo", "the onion", "all right then", "hey marian"]

N_BACKGROUND_CLIPS = 3_000  # AudioSet clips mixed in as background noise
N_RIRS = None               # room impulse responses to use (None = all ~270)
FULL_NEGATIVES = True       # download the 16 GB general-audio features (needed for a good model)

## 2. Install
openWakeWord is pinned to v0.6.0, the version Orion runs, so the model is compatible.

In [ ]:
!apt-get -qq install -y espeak-ng > /dev/null
![ -d openWakeWord ] || (git clone -q https://github.com/dscripka/openWakeWord && cd openWakeWord && git checkout -q v0.6.0)
![ -d piper-sample-generator ] || (git clone -q https://github.com/dscripka/piper-sample-generator && cd piper-sample-generator && git checkout -q f1988a4d54ed)
![ -f piper-sample-generator/models/en-us-libritts-high.pt ] || wget -q -O piper-sample-generator/models/en-us-libritts-high.pt https://github.com/rhasspy/piper-sample-generator/releases/download/v1.0.0/en-us-libritts-high.pt
!pip -q install -e ./openWakeWord --no-deps
# scipy<1.15: the `acoustics` package used for augmentation needs scipy.special.sph_harm (removed in 1.15).
!pip -q install mutagen torchinfo "torchmetrics<1" "speechbrain<1" "audiomentations<1" "torch-audiomentations<1" acoustics pronouncing "datasets<3" deep-phonemizer==0.0.19 espeak-phonemizer webrtcvad pyyaml onnx onnxruntime soundfile "scipy<1.15"

## 3. Download room echoes, background audio and negative features

In [ ]:
import io, os
import numpy as np
import scipy.io.wavfile
import soundfile as sf
from scipy.signal import resample_poly
from huggingface_hub import hf_hub_download

def save_wav(path, audio, sr=16000):
    audio = np.clip(np.asarray(audio, dtype=np.float32), -1, 1)
    scipy.io.wavfile.write(path, sr, (audio * 32767).astype(np.int16))

# Room impulse responses (MIT, 16 kHz)
os.makedirs("mit_rirs", exist_ok=True)
if len(os.listdir("mit_rirs")) < 10:
    from datasets import load_dataset
    rirs = load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)
    for i, row in enumerate(rirs):
        if N_RIRS is not None and i >= N_RIRS:
            break
        save_wav(f"mit_rirs/rir_{i:04d}.wav", row["audio"]["array"], row["audio"]["sampling_rate"])
print(len(os.listdir("mit_rirs")), "room impulse responses")

# Background audio: AudioSet clips (FLAC inside parquet), resampled to 16 kHz
os.makedirs("background_clips", exist_ok=True)
if len(os.listdir("background_clips")) < min(N_BACKGROUND_CLIPS, 100):
    import pyarrow.parquet as pq
    saved, part = 0, 0
    while saved < N_BACKGROUND_CLIPS:
        path = hf_hub_download("agkphysics/AudioSet", f"data/bal_train/{part:02d}.parquet",
                               repo_type="dataset", local_dir="hf")
        table = pq.read_table(path, columns=["audio"])
        for row in table.column("audio").to_pylist():
            if saved >= N_BACKGROUND_CLIPS:
                break
            try:
                audio, sr = sf.read(io.BytesIO(row["bytes"]), dtype="float32", always_2d=True)
            except Exception:
                continue
            audio = audio.mean(axis=1)
            if sr != 16000:
                g = np.gcd(16000, sr)
                audio = resample_poly(audio, 16000 // g, sr // g)
            if len(audio) >= 16000:
                save_wav(f"background_clips/bg_{saved:05d}.wav", audio)
                saved += 1
        os.remove(path)          # free disk; each parquet holds ~500 clips
        part += 1
print(len(os.listdir("background_clips")), "background clips")

# Pre-computed openWakeWord features: general audio (negatives) and false-positive validation
for name in (["openwakeword_features_ACAV100M_2000_hrs_16bit.npy"] if FULL_NEGATIVES else []) + ["validation_set_features.npy"]:
    if not os.path.exists(name):
        hf_hub_download("davidscripka/openwakeword_features", name, repo_type="dataset", local_dir=".")
NEGATIVE_FEATURES = ("openwakeword_features_ACAV100M_2000_hrs_16bit.npy" if FULL_NEGATIVES
                     else "validation_set_features.npy")
print("negative features:", NEGATIVE_FEATURES)

# openWakeWord's own feature models (melspectrogram + embedding)
import openwakeword.utils
openwakeword.utils.download_models(model_names=["embedding_model", "melspectrogram"])

## 4. Training config

In [ ]:
import yaml
config = {
    "model_name": MODEL_NAME,
    "target_phrase": TARGET_PHRASES,
    "custom_negative_phrases": NEGATIVE_PHRASES,
    "n_samples": N_SAMPLES,
    "n_samples_val": N_SAMPLES_VAL,
    "tts_batch_size": 50,
    "augmentation_batch_size": 16,
    "piper_sample_generator_path": "./piper-sample-generator",
    "output_dir": "./output",
    "rir_paths": ["./mit_rirs"],
    "background_paths": ["./background_clips"],
    "background_paths_duplication_rate": [1],
    "false_positive_validation_data_path": "./validation_set_features.npy",
    "augmentation_rounds": 1,
    "feature_data_files": {"ACAV100M_sample": "./" + NEGATIVE_FEATURES},
    "batch_n_per_class": {"ACAV100M_sample": 1024, "adversarial_negative": 50, "positive": 50},
    "model_type": "dnn",
    "layer_size": 32,
    "steps": STEPS,
    "max_negative_weight": 1500,
    "target_false_positives_per_hour": TARGET_FP_PER_HOUR,
}
with open("hey_orion.yml", "w") as f:
    yaml.safe_dump(config, f, sort_keys=False)
print(open("hey_orion.yml").read())

## 5. Two compatibility fixes
- **TTS checkpoint:** PyTorch 2.6+ refuses to load pickled checkpoints by default. The file's SHA-256 is checked against the official rhasspy release first, and only then loaded with `weights_only=False`.
- **TensorFlow Lite:** Orion uses the ONNX model. The TFLite step needs `tensorflow-cpu==2.8.1`, which no longer installs on Colab, so it's removed.

In [ ]:
import hashlib
TTS = "piper-sample-generator/models/en-us-libritts-high.pt"
TTS_SHA256 = "5258f21adf77c29248d9a89d51ca9492f6e75a498140584dcbed0edcab81960a"
h = hashlib.sha256(open(TTS, "rb").read()).hexdigest()
assert h == TTS_SHA256, f"TTS checkpoint hash mismatch ({h}); delete it and re-run the install cell"
g = "piper-sample-generator/generate_samples.py"
src = open(g).read()
if "weights_only=False" not in src:
    open(g, "w").write(src.replace("torch.load(model_path)", "torch.load(model_path, weights_only=False)"))
print("TTS checkpoint verified")

p = "openWakeWord/openwakeword/train.py"
src = open(p).read()
if "ONNX only for Orion" not in src:          # safe to re-run
    start = src.index("        convert_onnx_to_tflite(")
    end = src.index(")\n", start) + 2
    src = src[:start] + "        pass  # ONNX only for Orion (TFLite conversion removed)\n" + src[end:]
    open(p, "w").write(src)
print("TFLite conversion disabled")

## 6. Generate synthetic clips
The longest step: tens of thousands of TTS clips.

In [ ]:
!python openWakeWord/openwakeword/train.py --training_config hey_orion.yml --generate_clips

## 7. Add room echo and background noise, compute features

In [ ]:
!python openWakeWord/openwakeword/train.py --training_config hey_orion.yml --augment_clips

## 8. Train

In [ ]:
!python openWakeWord/openwakeword/train.py --training_config hey_orion.yml --train_model

## 9. Check the model and download it

In [ ]:
import glob, numpy as np, scipy.io.wavfile
from openwakeword.model import Model

onnx_path = f"output/{MODEL_NAME}.onnx"
m = Model(wakeword_models=[onnx_path], inference_framework="onnx")

def max_score(path):
    m.reset()
    sr, x = scipy.io.wavfile.read(path)
    x = np.concatenate([np.zeros(16000, np.int16), x.astype(np.int16), np.zeros(8000, np.int16)])
    x = x[: len(x) // 1280 * 1280]
    return max(max(m.predict(f).values()) for f in x.reshape(-1, 1280))

pos = sorted(glob.glob(f"output/{MODEL_NAME}/positive_test/*.wav"))[:200]
neg = sorted(glob.glob(f"output/{MODEL_NAME}/negative_test/*.wav"))[:200]
p = np.array([max_score(f) for f in pos]); n = np.array([max_score(f) for f in neg])
for t in (0.3, 0.5, 0.7):
    print(f"threshold {t}: detects {np.mean(p >= t):.0%} of held-out 'hey orion' clips, "
          f"fires on {np.mean(n >= t):.0%} of similar-sounding phrases")
print(onnx_path, os.path.getsize(onnx_path) // 1024, "KB")
try:
    from google.colab import files
    files.download(onnx_path)
except ImportError:
    pass

## 10. Use it in Orion

1. Copy the downloaded `hey_orion.onnx` to `~/orion/models/openwakeword/hey_orion.onnx`.
2. Test it with your own voice before switching: `python scripts/wakeword_test.py`.
3. In `config.yaml`: `models: [models/openwakeword/hey_orion.onnx, hey_jarvis]` to use both while you
   get used to it, then just the new one. Full details: `docs/wake-word.md`.